# Electricity Agreement Optimizer
Compare provider agreements against 12 months of electricity usage. Each company gets a specialist agent; the supervisor checks all results before a plan enters the ranking.

**Start in demo mode.** Demo contracts and negotiation questions are fictional fixtures, not GPT-5 outputs. Live mode calls GPT-5 for extraction and independent review.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "electricity_ao").exists():
    ROOT = ROOT / "Electricity_AO_Update"
if not (ROOT / "electricity_ao").exists():
    raise RuntimeError("Open Jupyter from Electricity_AO_Update or its parent folder")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from electricity_ao.agents import OpenAIBackend
from electricity_ao.demo import create_demo
from electricity_ao.documents import load_usage
from electricity_ao.models import Preferences, ProviderConfig
from electricity_ao.workflow import build_graph

## 1. Select inputs
Leave `MODE = "demo"` for an API-free run. For real comparisons, follow README setup, configure PDFs in `data/private/providers.json`, supply usage, then choose `"live"`. Each provider is a separate graph node. Switching cost is a one-time current-contract exit/setup cost applied to each candidate.

In [ ]:
MODE = "demo"
PROVIDERS_PATH = ROOT / "data/private/providers.json"
USAGE_PATH = ROOT / "data/private/usage.csv"
preferences = Preferences(
    max_term_months=24,
    minimum_renewable_percent=0,
    switching_cost_usd=0,
)

if MODE == "demo":
    providers, backend = create_demo(ROOT)
    usage = load_usage(ROOT / "data/usage_example.csv")
elif MODE == "live":
    providers = [ProviderConfig.model_validate(row)
                 for row in json.loads(PROVIDERS_PATH.read_text(encoding="utf-8"))]
    for provider in providers:
        provider.contract_path = str(ROOT / provider.contract_path)
    usage = load_usage(USAGE_PATH)
    backend = OpenAIBackend()
else:
    raise ValueError("MODE must be demo or live")

display(pd.DataFrame([month.model_dump() for month in usage.months]))
display(pd.DataFrame([provider.model_dump() for provider in providers]))

## 2. Build the supervisor graph
Specialists execute in parallel. The final supervisor waits for every provider, checks evidence and deterministic estimates, then independently reviews the source documents in live mode.

In [ ]:
graph = build_graph(providers, backend, mode=MODE)
print(graph.get_graph().draw_mermaid())

## 3. Run extraction and review
Live mode sends contract text to OpenAI and makes billable calls. Failed or unsupported plans remain visible in the review table and are excluded from ranking.

In [ ]:
state = graph.invoke({"usage": usage, "preferences": preferences, "results": []})
report = state["report"]
display(pd.DataFrame([review.model_dump() for review in report.reviews]))
print("Recommended provider:", report.recommended_provider_id or "None — resolve review issues")

## 4. Inspect the extracted terms and evidence
Validate rates, delivery fees, credits, renewal language, and page quotations before relying on live output.

In [ ]:
for result in sorted(state["results"], key=lambda result: result.provider_id):
    print("\nPROVIDER:", result.provider_id)
    if result.terms:
        display(pd.DataFrame([result.terms.model_dump(exclude={"evidence"})]))
        display(pd.DataFrame([item.model_dump() for item in result.terms.evidence]))
    else:
        print(result.issues)

## 5. Compare approved plans
Amounts are pre-tax estimates using the same historical usage. This is a lowest-cost comparison subject to your preferences, not a guarantee of future bills.

In [ ]:
ranking = pd.DataFrame([{
    "provider": item.provider_id,
    "plan": item.plan_name,
    "annual_usd": item.annual_usd,
    "first_year_usd": item.first_year_usd,
    "effective_cents_per_kwh": item.effective_cents_per_kwh,
} for item in report.ranking])
display(ranking)
if report.ranking:
    monthly = pd.DataFrame({
        item.provider_id: [month.cost_usd for month in item.monthly]
        for item in report.ranking
    }, index=[month.month.strftime("%Y-%m") for month in usage.months])
    ax = monthly.plot(marker="o", figsize=(11, 4), ylabel="Estimated monthly cost (USD, pre-tax)",
                      xlabel="Historical usage month", title="Your usage under each approved plan")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 6. Negotiation questions and limitations
These are suggested questions to discuss with a provider; no agreement is changed or message sent.

In [ ]:
for question in report.negotiation_questions:
    print("•", question)
for limitation in report.limitations:
    print("Note:", limitation)

## 7. Export your comparison
Outputs may contain contract terms. The outputs folder is excluded from Git.

In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
(output_dir / "comparison.json").write_text(report.model_dump_json(indent=2), encoding="utf-8")
ranking.to_csv(output_dir / "ranking.csv", index=False)
(output_dir / "extractions.json").write_text(
    json.dumps([result.model_dump(mode="json") for result in state["results"]], indent=2),
    encoding="utf-8",
)
print("Saved comparison.json, ranking.csv, and extractions.json to", output_dir)